<a href="https://colab.research.google.com/github/dr-bankert-augustana/PHYS_200/blob/main/Lesson_5_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a name="Notebook-Start"></a>

---

<font size = 7> <b> Lesson 5.3 () </b> </font>

---

<font size = 5> <b> Notebook Index </b> </font>

1. [Learning Outcomes](#Learning-Outcomes)

2. [Introduction](#Introduction)

3. [Adding Drag Force](#DragForce)
$
\newcommand{\lsum}{\displaystyle \sum\limits_{i=1}^{N}}
\newcommand{\parens}[1]{\left(#1\right)}
\newcommand{\dsfrac}[2]{\displaystyle\frac{#1}{#2}}
\newcommand{\dpfrac}[2]{\displaystyle\parens{\frac{#1}{#2}}}
\newcommand{\parderiv}[2]{\dsfrac{\partial #1}{\partial #2}}
\newcommand{\spc}{\hspace{0.1 pc}}
\newcommand{\ra}{\Rightarrow}
\newcommand{\of}[1]{{\scriptsize (#1)}}
\newcommand{\rule}{\Huge \hspace{-0.2 pc} \displaystyle\frac{\hspace{20 pc}}{\hspace{20 pc}}}
\newcommand{\mps}{\spc \frac{\textrm{m}}{\textrm{s}}}
\newcommand{\mpss}{\spc \frac{\textrm{m}}{\textrm{s}^2}}
$

<a name="Learning-Outcomes"></a>

---

#<font size = 6> <b> 1. Learning Outcomes </b> </font>

---

<font size = 5> <b> Learning Outcomes: </b> </font>

By the end of this lesson, students will be able to:

  <br>

  1. <b>Explain</b>

[Return to Top](#Notebook-Start)

<a name="Introduction"></a>

---

#<font size = 6> <b> 2. Introduction </b> </font>

---

##<font size = 5> <b> 2.1 Recap Free Fall in Vacuum</b> </font>

In the previous lessons of Unit 5, we simulated an object falling in a vacuum, that is, without air resistance. But the computational framework we used is
very general; it is relatively uncomplicated to add additional forces, including drag. In this lesson, we'll do just that!

<br>

To begin, let's implement the code that we have written so far.

<br>

In [ ]:
##=============================================================================================##
## Import Libraries:                                                                           ##
##=============================================================================================##

import time as tm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display_html

<font size = 5><b>Define Useful Functions</b></font>

In [ ]:
#@title This cell defines the functions: display_dataframes, create_model, display_model, and plot_data

##=============================================================================================##
## Function:  display_dataframes                                                               ##
##                                                                                             ##
## Purpose:   Display multiple DataFrames side-by-side with titles                             ##
##                                                                                             ##
## Input(s):  dataframe_list - List of DataFrames to be displayed                              ##
##            title_list     - List of titles for the displayed DataFrames                     ##
##            n_items        - Number of items to display (optional, default = 5)              ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_dataframes(dataframe_list, title_list, n_items = 5, tail = False):

  ##===========================================================================================##
  ## Create a String for Housing the Commands to Be Sent to the display_html() Function:       ##
  ##===========================================================================================##

  html_str = ""

  ##===========================================================================================##
  ## Loop Over the Elements in the item_list and title_list:                                   ##
  ##===========================================================================================##

  for df, title in zip(dataframe_list, title_list):

    # Convert the current dataframe info to html:

    if (tail == True):

      html_df = pd.DataFrame(df).tail(n_items).to_html()

    else:

      html_df = pd.DataFrame(df).head(n_items).to_html()

    ##=========================================================================================##
    ## Wrap title and DataFrames in a Styled HTML <div>:                                       ##
    ##=========================================================================================##

    html_str += "<div style='display: inline-block; margin-right: 20px; vertical-align: top;'>"

    html_str += "<h3 style='text-align: center;'>" + str(title) + "</h3><hr>" + str(html_df)

    html_str += "</div>"

  ##===========================================================================================##
  ## Send the HTML String to the display_html() Function:                                      ##
  ##===========================================================================================##

  display_html(html_str, raw = True)

##=============================================================================================##
## Function:  create_model                                                                     ##
##                                                                                             ##
## Purpose:   Display models' parameters and loss in a DataFrame                               ##
##                                                                                             ##
## Input(s):  model_list - List of models' names to be displayed                               ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def create_model(X, y, name, bias_on = True):

  ##===========================================================================================##
  ## Use a LinearRegression Object to Find the Best Fit for the Model:                         ##
  ##===========================================================================================##

  # Create a LinearRegression object with a forced-origin intercept:

  model = LinearRegression(fit_intercept = bias_on)

  # Fit the LinearRegression objects to the features and target:

  model.fit(X, y)

  # Get the coefficient and bias for the model:

  coefs = np.round(model.coef_[0], 2)
  bias  = np.round(model.intercept_, 2)

  ##===========================================================================================##
  ## Get The Model Predictions:                                                                ##
  ##===========================================================================================##

  predictions = model.predict(X)

  ##===========================================================================================##
  ## Calculate Model Losses:                                                                   ##
  ##===========================================================================================##

  loss = np.sqrt(mean_squared_error(y, predictions))

  ##===========================================================================================##
  ## Create a DataFrame to Store the Model's Results and Predictions:                          ##
  ##===========================================================================================##

  # Create the results DataFrame:

  results_df = pd.DataFrame({"Name": name, "Coefs": [coefs], "Bias": bias, "Loss": loss})

  # Create the predictions DataFrame:

  predictions_df = pd.DataFrame({"Target": X.iloc[:, 0]})

  predictions_df["Predictions"] = predictions

  ##===========================================================================================##
  ## Return the Model's Results and Predictions:                                               ##
  ##===========================================================================================##

  return results_df, predictions_df

##=============================================================================================##
## Function:  display_models                                                                   ##
##                                                                                             ##
## Purpose:   Display models' parameters and loss in a DataFrame                               ##
##                                                                                             ##
## Input(s):  model_list - List of models' names to be displayed                               ##
##            coef_list  - List of models' coefficients to be displayed                        ##
##            bias_list  - List of models' bias to be displayed                                ##
##            loss_list  - List of models' loss to be displayed                                ##
##            title      - Title for the display of models                                     ##
##            trunc      - Number of decimals to display for numbers (optional, default = 3)   ##
##            n_items    - Number of items to display (optional, default = 5)                  ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_model(model_list, coef_list, bias_list, loss_list, title, trunc = 3, n_items = 5):

  ##===========================================================================================##
  ## Round the Numeric Values to the Desired Level of Desired Truncation:                      ##
  ##===========================================================================================##

  for i in range (0, len(model_list)):

    coef_list[i] = np.round(coef_list[i], trunc)
    bias_list[i] = np.round(bias_list[i], trunc)
    loss_list[i] = np.round(loss_list[i], trunc)

  ##===========================================================================================##
  ## Create a DataFrame to Hold the Results:                                                   ##
  ##===========================================================================================##

  results = pd.DataFrame()

  ##===========================================================================================##
  ## Add the Contents of the DataFrame Columns:                                                ##
  ##===========================================================================================##

  # Add the model names:

  results["Model"] = model_list

  # Add the model coefficients:

  results["Coefficient(s)"] = coef_list

  # Add the model biases:

  results["Bias"] = bias_list

  # Add the model rmses:

  results["Loss"] = loss_list

  ##===========================================================================================##
  ## Index the Results DataFrame By Model Name:                                                ##
  ##===========================================================================================##

  results.set_index("Model", inplace = True)

  ##===========================================================================================##
  ## Display the Results DataFrame Using display_dataframes():                                 ##
  ##===========================================================================================##

  display_dataframes([results], [title], n_items)

##=============================================================================================##
## Function:  plot_data                                                                        ##
##                                                                                             ##
## Purpose:   Create a scatterplot with optional model overlays and error bands                ##
##                                                                                             ##
## Input(s):  x_data        - List of data points' x-axis values                               ##
##            y_data        - List of data points' y-axis values                               ##
##            title         - Graph title                                                      ##
##            axis_labels   - Override axis labels [x_label, y_label] (optional)               ##
##            model_list    - List of model predictions to overlay (optional)                  ##
##            color_list    - Colors for each model line (optional)                            ##
##            label_list    - Labels for each model line (optional)                            ##
##            error_display - Show +/- error band around first model (default is False)        ##
##            error         - Error value for shaded band (optional)                           ##
##                                                                                             ##
## Output(s): graph       - Matplotlib axes object, can be used for overplotting               ##
##=============================================================================================##

def plot_data(x_data, y_data, title, axis_labels = [], model_list = [], color_list = [],
              label_list = [], error_display = False, error = 0):

  ##===========================================================================================##
  ## Setup the Graph:                                                                          ##
  ##===========================================================================================##

  # Create the Matplotlib figure:

  figure = plt.figure(figsize = (12, 9))

  # Add a graph to the figure:

  graph = figure.add_subplot()

  # Set the graph background Color:

  graph.set_facecolor('lightcyan')

  # Set the graph title:

  graph.set_title(title, fontsize = 20)

  # Set the x_label and y_label:

  if (axis_labels != []):

    graph.set_xlabel(axis_labels[0], fontsize = 14)

    graph.set_ylabel(axis_labels[1], fontsize = 14)

  # Apply a grid to the graph:

  graph.grid(which = 'both')

  # Adjust the x-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'x', tight = False)

  # Adjust the y-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'y', tight = False)

  ##===========================================================================================##
  ## Add Data to the Graph:                                                                    ##
  ##===========================================================================================##

  # Create a scatterplot of the data:

  sns.scatterplot(x = x_data, y = y_data, ax = graph)

  # Overlay model predictions:

  for i in range(0, len(model_list)):

    graph.plot(model_list[i]["Target"], model_list[i]["Predictions"], color = color_list[i],
               label = label_list[i])

  ##===========================================================================================##
  ## If requested, show the +/- error bounds:                                                  ##
  ##===========================================================================================##

  if ((error_display == True) and model_list != []):

    # Create the error+ model:

    model_plus_error  = model_list[-1]["Predictions"] + error

    # Create the error- model:

    model_minus_error = model_list[-1]["Predictions"] - error

    # Store the error+ and error- models in a DataFrame and Sort by x_data values:

    error_df = pd.DataFrame({
        'X-Data': model_list[0]["Target"],
        'Model+Error': model_plus_error,
        'Model-Error': model_minus_error
    }).sort_values(by = "X-Data")

    # Use graph.fill to highlight the region between the error+ and error- models:

    graph.fill_between(error_df['X-Data'], error_df['Model+Error'], error_df['Model-Error'],
                       alpha = 0.5, color = (0.6, 0.6, 0.6), label = "Error Bounds")

  ##===========================================================================================##
  ## Apply the Legend and Return the graph Object:                                             ##
  ##===========================================================================================##

  # Add the graph legend:

  if (label_list != []): graph.legend()

  # Return the graph:

  return graph

In [ ]:
##=============================================================================================##
## Set the Simulation Parameters:                                                              ##
##=============================================================================================##

# Local acceleration due to gravity (in m/s^2):

g = 9.8

# Time step size (in seconds):

dt = 0.01

# Simulation end time (in seconds):

t_end = 100

##=============================================================================================##
## Store the parameters in a Pandas Series:                                                    ##
##=============================================================================================##

parameters_gravity_only = pd.Series({
    "g":  g,
    "dt": dt,
    "t_end": t_end
})

# Rename the columns:

parameters_gravity_only.name = 'Value'

parameters_gravity_only.index.name = "Parameter"

##=============================================================================================##
## Create the State Object                                                                     ##
##=============================================================================================##

# Initial Height:

y_initial = 381.0 # meters

# Initial Velocity:

v_initial = 0 # meters per second

# Initial Time:

t_initial = 0 # seconds

##=============================================================================================##
## Store the initial state in a Pandas Series:                                                 ##
##=============================================================================================##

initial_state = pd.Series({
    "Position" : y_initial,
    "Velocity" : v_initial,
    "Time"     : t_initial
})

initial_state.name = 'Value'

initial_state.index.name = "State"

##=============================================================================================##
## Display the Parameters and Initial State:                                                   ##
##=============================================================================================##

display_dataframes([parameters_gravity_only, initial_state],
                   ["Gravity Only Parameters", "Initial State"])

<font size = 5><b>Define the Acceleration Function For Gravity Only</b></font>

In [ ]:
##=============================================================================================##
## Define the Acceleration Function For the Falling Object Simulation:                         ##
##=============================================================================================##

def acceleration_gravity_only_1d(state, parameters):

  ##===========================================================================================##
  ## Unpack the state and parameters:                                                          ##
  ##===========================================================================================##

  # Get the local acceleration of gravity:

  g = parameters["g"]

  ##===========================================================================================##
  ## Return the Acceleration:                                                                  ##
  ##===========================================================================================##

  return -g

<font size = 5><b>Create the Various Change Functions</b></font>

In [ ]:
#@title This cell defines the Euler, Euler-Cromer, Euler-Richardson, and Runge-Kutta 4th Order DEQ Solver Algorithms.

##=============================================================================================##
## Define the 1D Euler Algorithm Change Function:                                              ##
##=============================================================================================##

def cf_euler_1d(state, parameters, acceleration):

  ##===========================================================================================##
  ## Prepare the Algorithm Values:                                                             ##
  ##===========================================================================================##

  # Create a local copy of the state object:

  state_local = state.copy()

  # Get the time step size:

  dt = parameters['dt']

  # Get the current position:

  x = state_local['Position']

  # Get the current velocity:

  v = state_local['Velocity']

  # Get the current acceleration:

  a = acceleration(state_local, parameters)

  # Update the position:

  x += v * dt

  # Update the velocity:

  v += a * dt

  ##===========================================================================================##
  ## Update the State:                                                                         ##
  ##===========================================================================================##

  # Update the local state position:

  state_local['Position'] = x

  # Update the local state velocity:

  state_local['Velocity'] = v

  # Update the local state time:

  state_local['Time'] += dt

  ##===========================================================================================##
  ## Return the Updated State:                                                                 ##
  ##===========================================================================================##

  return state_local

##=============================================================================================##
## Define the 1D Euler-Cromer Algorithm Function:                                              ##
##=============================================================================================##

def cf_euler_cromer_1d(state, parameters, acceleration):

  ##===========================================================================================##
  ## Prepare the Algorithm Values:                                                             ##
  ##===========================================================================================##

  # Create a local copy of the state object:

  state_local = state.copy()

  # Get the time step size:

  dt = parameters['dt']

  # Get the current position:

  x = state_local['Position']

  # Get the current velocity:

  v = state_local['Velocity']

  # Get the current acceleration:

  a = acceleration(state_local, parameters)

  # Update the velocity:

  v += a * dt

  # Update the position:

  x += v * dt

  ##===========================================================================================##
  ## Update the State:                                                                         ##
  ##===========================================================================================##

  # Update the local state position:

  state_local['Position'] = x

  # Update the local state velocity:

  state_local['Velocity'] = v

  # Update the local state time:

  state_local['Time'] += dt

  ##===========================================================================================##
  ## Return the Updated State:                                                                 ##
  ##===========================================================================================##

  return state_local

##=============================================================================================##
## Define the 1D Euler Algorithm Function:                                                     ##
##=============================================================================================##

def cf_euler_richardson_1d(state, parameters, acceleration):

  ##===========================================================================================##
  ## Prepare the Algorithm Values:                                                             ##
  ##===========================================================================================##

  # Create a local copy of the state object:

  state_local = state.copy()

  # Get the time step size:

  dt = parameters['dt']

  # Get the current position:

  x = state_local['Position']

  # Get the current velocity:

  v = state_local['Velocity']

  # Get the current acceleration:

  a = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Get the Midpoint Values:                                                                  ##
  ##===========================================================================================##

  # Calculate the midpoint velocity:

  v_mid = v + a * dt / 2

  # Calculate the midpoint position:

  x_mid = x + v * dt / 2

  # Update the local state:

  state_local['Position'] = x_mid

  state_local['Velocity'] = v_mid

  # Calculate the midpoint acceleration:

  a_mid = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Calculate the Updated Values:                                                             ##
  ##===========================================================================================##

  # Update the position:

  x += v_mid * dt

  # Update the velocity:

  v += a_mid * dt

  ##===========================================================================================##
  ## Update the State:                                                                         ##
  ##===========================================================================================##

  # Update the local state position:

  state_local['Position'] = x

  # Update the local state velocity:

  state_local['Velocity'] = v

  # Update the local state time:

  state_local['Time'] += dt

  ##===========================================================================================##
  ## Return the Updated State:                                                                 ##
  ##===========================================================================================##

  return state_local

##=============================================================================================##
## Define the 1D Runge-Kutta Change Function:                                                  ##
##=============================================================================================##

def cf_runge_kutta_1d(state, parameters, acceleration):

  # Create a local copy of the state object:

  state_local = state.copy()

  # Unpack the state:

  y = state_local['Position']

  v = state_local['Velocity']

  # Compute the acceleration:

  a = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Calculate the k1 Values (Slope at the Beginning of the Time Step):                        ##
  ##===========================================================================================##

  k1_y = v

  k1_v = a

  ##===========================================================================================##
  ## Calculate the k2 Values (Slope at the Midpoint of the Time Step, Based on k1 Values):     ##
  ##===========================================================================================##

  # Position k2 values:

  k2_y = v + 0.5 * k1_v * dt

  # Velocity k2 values:

  y_mid_1 = y + 0.5 * k1_y * dt

  v_mid_1 = v + 0.5 * k1_v * dt

  # Acceleration k2 values:

  state_local['Position'] = y_mid_1

  state_local['Velocity'] = v_mid_1

  k2_v = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Calculate the k3 Values (Slope at the Midpoint of the Time Step, Based on k2 Values):     ##
  ##===========================================================================================##

  # Position k3 values:

  k3_y = v + 0.5 * k2_v * dt

  # Velocity k3 values:

  y_mid_2 = y + 0.5 * k2_y * dt

  v_mid_2 = v + 0.5 * k2_v * dt

  # Acceleration k3 values:

  state_local['Position'] = y_mid_2

  state_local['Velocity'] = v_mid_2

  k3_v = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Calculate the k4 Values (Slope at the End of the Time Step, Based on k3 Values):          ##
  ##===========================================================================================##

  # Position k4 values:

  k4_y = v + k3_v * dt

  # Velocity k4 values:

  y_end = y + k3_y * dt

  v_end = v + k3_v * dt

  # Acceleration k4 values:

  state_local['Position'] = y_end

  state_local['Velocity'] = v_end

  k4_v = acceleration(state_local, parameters)

  ##===========================================================================================##
  ## Update the State Values:                                                                  ##
  ##===========================================================================================##

  # Update the object's velocity:

  v += (k1_v + 2.0 * k2_v + 2.0 * k3_v + k4_v) * dt / 6.0

  # Update the object's position:

  y += (k1_y + 2.0 * k2_y + 2.0 * k3_y + k4_y) * dt / 6.0

  # Update the local state positions:

  state_local['Position'] = y

  # Update the local state velocity:

  state_local['Velocity'] = v

  # Update the local state time:

  state_local['Time'] += dt

  # Return the updated local state:

  return state_local

<font size = 5><b>Create the Run Simulation Function</b></font>

In [ ]:
##=============================================================================================##
## Create a Function to Set Up and Run the Simulation:                                         ##
##=============================================================================================##

def run_simulation(parameters, initial_state, change_function, acceleration):

  ##===========================================================================================##
  ## Get the Simulation Start Time:                                                            ##
  ##===========================================================================================##

  start_time = tm.time()

  ##===========================================================================================##
  ## Unpack the Simulation Parameters:                                                         ##
  ##===========================================================================================##

  # Get the end time:

  end_time = parameters.loc['t_end']

  # Get the step size:

  dt = parameters.loc['dt']

  ##===========================================================================================##
  ## Create a Copy of the Initial State:                                                       ##
  ##===========================================================================================##

  state = initial_state.copy()

  ##===========================================================================================##
  ## Create the System's State History:                                                        ##
  ##===========================================================================================##

  # Create the state_history DataFrame:

  state_history = pd.DataFrame()

  state_history.name = 'Motion Over Time'

  state_history.index.name = 'Time Step'

  # Set the first entry in the state_history to be the initial system state:

  state_history[0] = state

  state_history = state_history.T

  ##===========================================================================================##
  ## Run the Simulation for the Desired Number of Time Steps:                                  ##
  ##===========================================================================================##

  end_condition = False

  while (end_condition == False):

    # Get the current time step:

    i = state_history.index[-1]

    # Get the state of the system for the current time step:

    state = change_function(state, parameters, acceleration)

    # Store the new system state in the state_history DataFrame:

    state_history.loc[i + 1] = state.T

    # If an end condition has been reached, break the loop:

    if (state['Time'] >= end_time): end_condition = True

    if (state['Position'] <= 0.0): end_condition = True

  ##===========================================================================================##
  ## Get the Simulation End Time:                                                              ##
  ##===========================================================================================##

  end_time = tm.time()

  print("Elapsed Simulation Runtime:", np.round(end_time - start_time, 5), "seconds")

  ##===========================================================================================##
  ## Return the System's State History:                                                        ##
  ##===========================================================================================##

  return state_history

<font size = 5><b>Create the Loss Function<b></font>

In [ ]:
##=============================================================================================##
## Create a Function to Calculate Loss:                                                        ##
##=============================================================================================##

def loss_1d(parameters, state_history, acceleration):

  ##===========================================================================================##
  ## Calculate the Initial Total Energy:                                                       ##
  ##===========================================================================================##

  total_energy_initial = parameters['g'] * state_history.iloc[0]['Position']

  ##===========================================================================================##
  ## Calculate the Total Energy History:                                                       ##
  ##===========================================================================================##

  total_energy_history = parameters['g'] * state_history.iloc[:]['Position'] + \
                         0.5 * state_history.iloc[:]['Velocity'] ** 2

  ##===========================================================================================##
  ## Calculate the Energy Lost to Drag Force (Work is force times velocity * dt):              ##
  ##===========================================================================================##

  v  = state_history.iloc[:]['Velocity']
  dt = parameters['dt']

  drag_acc = (acceleration(state_history, parameters) + parameters['g'])

  drag_work = (np.abs(v * drag_acc) * dt).cumsum()

  ##===========================================================================================##
  ## Calculate the Loss History:                                                               ##
  ##===========================================================================================##

  loss_history = (total_energy_history - total_energy_initial + drag_work) / total_energy_initial

  ##===========================================================================================##
  ## Return the Loss History:                                                                  ##
  ##===========================================================================================##

  return loss_history

<font size = 5><b>Create the Analyze Simulation Function<b></font>

In [ ]:
##=============================================================================================##
## Create a Function to Analyze the Simulation Results:                                        ##
##=============================================================================================##

def analyze_simulation(state_history, parameters, loss_function, acceleration, x_list = [],
                       y_list = [], color_list = [], title_list = [], x_label_list = [],
                       y_label_list = [], aspect_list = [], plot_data = False, display_data = False):

  ##===========================================================================================##
  ## Compute the Simulation_Loss:                                                              ##
  ##===========================================================================================##

  loss_history = loss_function(parameters, state_history, acceleration)

  ##===========================================================================================##
  ## Add the Loss History to the State History:                                                ##
  ##===========================================================================================##

  state_history['Loss'] = loss_history

  ##===========================================================================================##
  ## Calculate the Total Loss:                                                                 ##
  ##===========================================================================================##

  total_loss = loss_history.sum()

  ##===========================================================================================##
  ## If requested, display the history data:                                                   ##
  ##===========================================================================================##

  if (display_data == True):

    # View the system's state history and final state:

    display_dataframes([state_history, state_history.iloc[-1]],["State History", "Final State"],
                       n_items = 7)

  ##===========================================================================================##
  ## If requested, plot the history data:                                                      ##
  ##===========================================================================================##

  if (plot_data == True):

    # Get the settings for each graph to be made:

    for X, Y, color, title, x_label, y_label, aspect in \
      zip(x_list, y_list, color_list, title_list, x_label_list, y_label_list, aspect_list):

      # Create the graph:

      x_size = 10
      y_size = 10 * aspect

      ax = state_history.plot(
        kind    = 'line',
        y       = [Y],
        x       = str(X),
        color   = [color],
        figsize = (x_size, y_size)
      )

      # Set the graph title:

      ax.set_title(title, fontsize = 14)

      # Set the Graph x-axis label:

      ax.set_xlabel(x_label, fontsize = 12)

      # Set the Graph y-axis label:

      ax.set_ylabel(y_label, fontsize = 12)

      # Add grid and suppress the legend:

      ax.grid(True, linestyle = '--', alpha = 0.5)

      legend = ax.legend()

      legend.remove()

    # Display the plots:

    plt.show()

  ##===========================================================================================##
  ## Return the History Data:                                                                  ##
  ##===========================================================================================##

  return state_history.iloc[-1]['Loss']

<font size = 5><b>Run the Gravity Only Falling Object Simulation<b></font>

In [ ]:
##=============================================================================================##
## Run the Simulation and Get the Sate History:                                                ##
##=============================================================================================##

# Set the desired parameter set:

parameters = parameters_gravity_only

# Set the desired acceleration function:

acceleration = acceleration_gravity_only_1d

# Set the desired change function:

change_function = cf_euler_1d

# Run the simulation:

history_fall = run_simulation(parameters, initial_state, change_function, acceleration)

##=============================================================================================##
## Analyze the Sate History and Get the Simulion Loss:                                         ##
##=============================================================================================##

# Set the desired loss function:

loss_falling_object = loss_1d

# Set the plotting parameters:

y_list   = ['Position', 'Velocity', 'Loss']
x_list   = ['Time', 'Time', 'Time']
colors   = ['blue', 'red', 'green']
titles   = ['Height Vs Time', 'Velocity Vs Time', 'Loss Vs Time']
y_labels = ['Height (m)', 'Velocity (m/s)', 'Loss']
x_labels = ['Time (s)', 'Time (s)', 'Time (s)']
aspects  = [0.6, 0.6, 0.6]

# Run the analyze simulation function:

loss = analyze_simulation(history_fall, parameters, loss_1d, acceleration,
                          x_list, y_list, colors, titles, x_labels, y_labels, aspects,
                          plot_data = True, display_data = True)

[Return to Top](#Notebook-Start)

<a name="Drag"></a>

---

#<font size = 6> <b> 3. Adding Drag Force </b> </font>

---

##<font size = 5> <b> 3.1 The Physics of Drag </b> </font>

<br>

<b>The drag force equation</b>

As an object moves through a fluid, like air, the object applies force
to the air and, in accordance with the third law of motion, the air applies an equal and opposite force to the object.

<br>

Drag is a "reactive force", which means that it always acts in the direction opposite to the direction of
travel.  Unlike friction, its magnitude is dependent on the velocity, and can be calculated using the drag equation:

$$F_d = \frac{1}{2}~\rho~v^2~C_d~A$$

where

-   $F_d$ is force due to drag, in newtons ($N$).

-   $\rho$ is the density of the fluid in $kg/m^3$.

-   $v$ is the magnitude of velocity in $m/s$.

-   $A$ is the the *projected frontal area* of the object in $m^2$. You can think of this as the visible area of the object as seen from a point on its line of travel.  So a falling cylinder might have an $A$ in the shape of a circle or a rectangle, depending on its orientation as it falls.

-   $C_d$ is the *drag coefficient*, a dimensionless quantity that depends on the shape of the object (including length but not frontal area), its surface properties, and how it interacts with the fluid.

<br>

Of course, the drag equation is itself a model, based on the assumption that $C_d$ does not depend on the other terms in the equation: density, velocity, and area. If the object is moving very fast, or spinning, or if $C_D$ depends on velocity or density, the model can be less effective.

<br>

##<font size = 5> <b> 3.2 Determining a drag coefficient </b> </font>

For objects moving at moderate speeds through air, typical drag
coefficients are between 0.1 and 1.0, with blunt objects at the high end of the range and streamlined objects at the low end.

<br>

<img src = https://github.com/MAugspurger/ModSimPy_MAugs/raw/main/Images_and_Data/Images/3_6/Drag_coef.PNG width = 500>

<br>

For simple geometric objects we can sometimes guess the drag coefficient with reasonable accuracy; for more complex objects we usually have to take measurements and estimate $C_d$ from data.

<br>

Since a falling penny would likely be rotating unpredictably, we should use measurements to estimate $C_d$. In particular, we can measure *terminal velocity*, $v_{term}$, which is
the speed where drag halts the objects' acceleration.  If we know this terminal velocity, we can use the second law of motion to find $C_d$, by naming our two forces (gravity and drag) setting the acceleration to 0:

<br>

$$\Sigma F = ma \qquad \rightarrow \qquad F_{drag} + F_{grav} =0$$

<br>

This leads to:

<br>

$$F_{drag} = - F_{grav}$$

<br>

$$\frac{1}{2}~\rho~v_{terminal}^2~C_d~A = - m g$$

<br>

where $m$ is the mass of the object and $g$ is acceleration due to gravity. Solving this equation for $C_d$ yields:

<br>

$$C_d = \frac{2~m g}{\rho~v_{terminal}^2~A}$$

<br>

The density of air depends on temperature, barometric pressure (which depends on
altitude), and humidity, among other things.  This value might be typical in New York City at 20 °C.   Our $C_d$ with these parameters is about $0.17$. We set the terminal velocity to 29 m/s.

## <b></b>
<font size = 5><b>Set the Falling Object Simulation Parameters</b></font>

In [ ]:
##=============================================================================================##
## Set the Simulation Parameters:                                                              ##
##=============================================================================================##

# Local acceleration due to gravity (in m/s^2):

g = 9.8

# Terminal velocity (in m/s):

v_term = 29.0

# Air mass density (in kg/m^3):

rho = 1.2

# Object mass (in kg):

mass = 0.0025       # mass

# Object diameter (in m):

diameter = 0.019

# Calculate the object's area:

area = np.pi * (diameter/2)**2

# Calculate the object's drag coeffcient:

C_d = (2.0 * mass * g) / (rho * v_term**2 * area)

print("The object's drag coefficient is: " + str(np.round(C_d, 3)))

# Time step size (in seconds):

dt = 0.01

# Simulation end time (in seconds):

t_end = 100

##=============================================================================================##
## Store the parameters in a Pandas Series:                                                    ##
##=============================================================================================##

parameters_drag = pd.Series({
    "g":  g,
    "Cd": C_d,
    "rho": rho,
    "mass": mass,
    "area": area,
    "dt": dt,
    "t_end": t_end
})

# Rename the columns:

parameters_drag.name = 'Value'

parameters_drag.index.name = "Parameter"

##=============================================================================================##
## Create the State Object                                                                     ##
##=============================================================================================##

# Initial Height:

y_initial = 381.0 # meters

# Initial Velocity:

v_initial = 0 # meters per second

# Initial Time:

t_initial = 0 # seconds

##=============================================================================================##
## Display the Parameters and Initial State:                                                   ##
##=============================================================================================##

display_dataframes([parameters_drag, initial_state],
                   ["Falling Object With Drag Parameters", "Initial State"])

##<font size = 5> <b> 3.3 Define the Acceleration Function </b> </font>

`f_drag` is force due to drag, based on the drag equation, and `a_drag` is
acceleration due to drag, based on the second law.

<br>

To compute total acceleration, we add accelerations due to gravity and
drag. Notice that the signs of `g` and `a_drag` indicate their directions. As usual, let's test the slope function with the initial conditions.

In [ ]:
##=============================================================================================##
## Define the Acceleration Function For the Falling Object Simulation:                         ##
##=============================================================================================##

def acceleration_air_drag_1d(state, parameters):

  ##===========================================================================================##
  ## Unpack the state and parameters:                                                          ##
  ##===========================================================================================##

  # Get the current velocity:

  v = state["Velocity"]

  # Get the local acceleration of gravity:

  g = parameters["g"]

  # Get the object's mass:

  m = parameters["mass"]

  # Get the air density:

  rho = parameters["rho"]

  # Get the object's area:

  area = parameters["area"]

  # Get the object's drag coefficient:

  C_d = parameters["Cd"]

  ##===========================================================================================##
  ## Calculate the Net Force on the Object:                                                    ##
  ##===========================================================================================##

  # Calculate the force due to local gravitational acceleration:

  f_grav = - m * g

  # Calculate the force due to air drag (force is always in the opposite direction of v):

  f_drag = - np.sign(v) * rho * v**2 * C_d * area / 2.0

  # Calculate the Net force:

  f_net = f_grav + f_drag

  ##===========================================================================================##
  ## Return the Acceleration:                                                                  ##
  ##===========================================================================================##

  return f_net / mass

<font size = 5><b>Run the Falling Object Simulation With Air Drag <b></font>

In [ ]:
##=============================================================================================##
## Run the Simulation and Get the Sate History:                                                ##
##=============================================================================================##

# Set the desired parameter set:

parameters = parameters_drag

# Set the desired acceleration function:

acceleration = acceleration_air_drag_1d

# Set the desired change function:

change_function = cf_euler_1d

# Run the simulation:

history_fall = run_simulation(parameters, initial_state, change_function, acceleration)

##=============================================================================================##
## Analyze the Sate History and Get the Simulion Loss:                                         ##
##=============================================================================================##

# Set the desired loss function:

loss_falling_object = loss_1d

# Set the plotting parameters:

y_list   = ['Position', 'Velocity', 'Loss']
x_list   = ['Time', 'Time', 'Time']
colors   = ['blue', 'red', 'green']
titles   = ['Height Vs Time', 'Velocity Vs Time', 'Loss Vs Time']
y_labels = ['Height (m)', 'Velocity (m/s)', 'Loss']
x_labels = ['Time (s)', 'Time (s)', 'Time (s)']
aspects  = [0.6, 0.6, 0.6]

# Run the analyze simulation function:

loss = analyze_simulation(history_fall, parameters, loss_1d, acceleration,
                          x_list, y_list, colors, titles, x_labels, y_labels, aspects,
                          plot_data = True, display_data = True)

The final height is close to 0, as expected.  Interestingly, the final velocity is not exactly terminal velocity, which is a reminder that the simulation results are only approximate.

<br>

With air resistance, it takes about 15 seconds for the penny to reach the sidewalk.

##<font size = 5> <b> 3.5 Simulation With Initial Velocity </b> </font>

Now, let's test our simulation with an initial velocity and see how it affects the outcome.


In [ ]:
##=============================================================================================##
## Run the Simulation and Get the Sate History:                                                ##
##=============================================================================================##

# Create a copy of the initial state:

initial_state_copy = initial_state.copy()

# Set the initial velocity (in meters / second):

initial_state_copy["Velocity"] = 10

# Set the desired parameter set:

parameters = parameters_drag

# Set the desired acceleration function:

acceleration = acceleration_air_drag_1d

# Set the desired change function:

change_function = cf_euler_1d

# Run the simulation:

history_fall = run_simulation(parameters, initial_state_copy, change_function, acceleration)

##=============================================================================================##
## Analyze the Sate History and Get the Simulion Loss:                                         ##
##=============================================================================================##

# Set the desired loss function:

loss_falling_object = loss_1d

# Set the plotting parameters:

y_list   = ['Position', 'Velocity', 'Loss']
x_list   = ['Time', 'Time', 'Time']
colors   = ['blue', 'red', 'green']
titles   = ['Height Vs Time', 'Velocity Vs Time', 'Loss Vs Time']
y_labels = ['Height (m)', 'Velocity (m/s)', 'Loss']
x_labels = ['Time (s)', 'Time (s)', 'Time (s)']
aspects  = [0.6, 0.6, 0.6]

# Run the analyze simulation function:

loss = analyze_simulation(history_fall, parameters, loss_1d, acceleration,
                          x_list, y_list, colors, titles, x_labels, y_labels, aspects,
                          plot_data = True, display_data = True)